# 🎓 Campus Compass — Placement Tracker
### Student Placement Management System | Python SL Project

**Run this notebook in Google Colab from top to bottom.**  
This beginner-friendly project demonstrates **lists, tuples, sets, dictionaries, Pandas, NumPy, and file handling**, with a clean dashboard-style interface.

**What it does**
- View placement KPIs and charts
- Search/filter student records
- Add or update placement details
- Export the current tracker to CSV
- Summarize offers, placed students, and company activity

> **Demo data:** The names and records below are fictional. Replace them with approved data. Do not upload private student information without permission.

In [ ]:
# 1. Imports and setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from datetime import datetime
import os

print("Campus Compass is ready ✨")

## 2. Python data structures
We intentionally use each required structure:
- **List:** placement records and filter choices
- **Tuple:** fixed tracker fields
- **Set:** unique companies and statuses
- **Dictionary:** KPI summary and lookup of student records

In [ ]:
# Fixed schema stored as a tuple
FIELDS = ("student_id", "name", "department", "graduation_year",
          "cgpa", "status", "company", "package_lpa")

# Demo records stored in a list of dictionaries
records = [
    {"student_id":"STU101","name":"Aarav Shah","department":"CSE","graduation_year":2027,"cgpa":8.7,"status":"Placed","company":"TechNova","package_lpa":8.5},
    {"student_id":"STU102","name":"Diya Patil","department":"IT","graduation_year":2027,"cgpa":9.1,"status":"Placed","company":"CloudPeak","package_lpa":10.0},
    {"student_id":"STU103","name":"Kabir Joshi","department":"ECE","graduation_year":2027,"cgpa":7.8,"status":"Interviewing","company":"ByteWorks","package_lpa":np.nan},
    {"student_id":"STU104","name":"Meera Rao","department":"CSE","graduation_year":2027,"cgpa":8.2,"status":"Applied","company":"—","package_lpa":np.nan},
    {"student_id":"STU105","name":"Ishaan Kulkarni","department":"IT","graduation_year":2027,"cgpa":7.4,"status":"Not Placed","company":"—","package_lpa":np.nan},
    {"student_id":"STU106","name":"Sara Khan","department":"AIML","graduation_year":2027,"cgpa":8.9,"status":"Placed","company":"TechNova","package_lpa":7.2},
    {"student_id":"STU107","name":"Vivaan Deshmukh","department":"ECE","graduation_year":2027,"cgpa":8.0,"status":"Applied","company":"—","package_lpa":np.nan},
    {"student_id":"STU108","name":"Anaya Singh","department":"CSE","graduation_year":2027,"cgpa":9.4,"status":"Placed","company":"DataSpring","package_lpa":12.0},
    {"student_id":"STU109","name":"Rehan Shaikh","department":"AIML","graduation_year":2027,"cgpa":7.9,"status":"Interviewing","company":"CloudPeak","package_lpa":np.nan},
    {"student_id":"STU110","name":"Tara Nair","department":"IT","graduation_year":2027,"cgpa":8.5,"status":"Placed","company":"ByteWorks","package_lpa":9.0},
]

# List of records; set of distinct departments/companies; dictionary lookup
department_options = sorted({r["department"] for r in records})  # set
company_options = sorted({r["company"] for r in records if r["company"] != "—"})
student_lookup = {r["student_id"]: r for r in records}           # dictionary

print("Schema tuple:", FIELDS)
print("Number of records:", len(records))
print("Departments (unique set converted to list):", department_options)
print("Companies:", company_options)
print("Lookup example:", student_lookup["STU101"])

## 3. Convert records into a Pandas DataFrame
Pandas handles sorting, filtering, and reporting. NumPy helps calculate numeric summaries.

In [ ]:
df = pd.DataFrame(records, columns=FIELDS)
df["cgpa"] = pd.to_numeric(df["cgpa"], errors="coerce")
df["package_lpa"] = pd.to_numeric(df["package_lpa"], errors="coerce")
display(df)

## 4. Dashboard overview
Placement rate here means **placed students ÷ total students**. It is a demo snapshot, not an official college statistic.

In [ ]:
def calculate_kpis(data):
    total = len(data)
    placed = int((data["status"] == "Placed").sum())
    placement_rate = (placed / total * 100) if total else 0
    avg_package = data.loc[data["status"] == "Placed", "package_lpa"].mean()
    return {
        "Total Students": total,
        "Placed": placed,
        "Placement Rate": round(placement_rate, 1),
        "Companies": int(data.loc[data["status"] == "Placed", "company"].nunique()),
        "Average Package (LPA)": round(float(avg_package), 2) if pd.notna(avg_package) else 0
    }

kpis = calculate_kpis(df)  # dictionary of dashboard metrics
display(HTML(f'''
<div style="font-family:Arial;background:linear-gradient(120deg,#101a35,#172d50);color:white;padding:22px;border-radius:18px">
  <div style="font-size:13px;letter-spacing:2px;color:#a8c7fa">CAMPUS COMPASS / PLACEMENT INTELLIGENCE</div>
  <h1 style="margin:8px 0 18px">Placement Dashboard</h1>
  <div style="display:flex;flex-wrap:wrap;gap:12px">
    {''.join(f'<div style="background:#233b5f;padding:16px 20px;border-radius:12px;min-width:135px;flex:1"><div style="font-size:12px;color:#bdd2ef">{k}</div><div style="font-size:27px;font-weight:bold;margin-top:7px">{v}{"%" if k=="Placement Rate" else ""}</div></div>' for k,v in kpis.items())}
  </div>
  <p style="color:#c6d6eb;margin-bottom:0">Demo dataset • Graduation cohort 2027</p>
</div>
'''))

In [ ]:
# Visual analytics (Matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
status_counts = df["status"].value_counts()
axes[0].bar(status_counts.index, status_counts.values)
axes[0].set_title("Students by placement status")
axes[0].set_ylabel("Student count")
axes[0].tick_params(axis="x", rotation=20)

placed_data = df[df["status"] == "Placed"].groupby("company").size().sort_values(ascending=False)
axes[1].bar(placed_data.index, placed_data.values)
axes[1].set_title("Placed students by company")
axes[1].set_ylabel("Student count")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print("NumPy package average CGPA:", round(float(np.nanmean(df["cgpa"].to_numpy())), 2))

## 5. Search and filter
Change the text or department below to explore records.

In [ ]:
def search_students(data, query="", department="All", status="All"):
    result = data.copy()
    if query.strip():
        q = query.strip().lower()
        result = result[result.apply(
            lambda row: q in str(row["name"]).lower()
                        or q in str(row["student_id"]).lower()
                        or q in str(row["company"]).lower(), axis=1)]
    if department != "All":
        result = result[result["department"] == department]
    if status != "All":
        result = result[result["status"] == status]
    return result

# Example search: filter CSE students who are placed
display(search_students(df, department="CSE", status="Placed"))

## 6. Add a student or update an existing record
Edit the values in the example call, or use the helper functions from your own interface.

In [ ]:
def upsert_student(data, student):
    # Validate required fields and avoid duplicate IDs
    missing = [field for field in FIELDS if field not in student]
    if missing:
        raise ValueError(f"Missing fields: {missing}")
    if not str(student["student_id"]).strip() or not str(student["name"]).strip():
        raise ValueError("Student ID and name are required.")
    if student["status"] not in {"Placed", "Applied", "Interviewing", "Not Placed"}:
        raise ValueError("Choose a valid placement status.")
    if not 0 <= float(student["cgpa"]) <= 10:
        raise ValueError("CGPA must be between 0 and 10.")
    if student["status"] == "Placed" and (not student["company"] or pd.isna(student["package_lpa"])):
        raise ValueError("For placed students, enter company and package.")
    updated = data.copy()
    sid = str(student["student_id"]).strip()
    student = {field: student[field] for field in FIELDS}
    student["student_id"] = sid
    mask = updated["student_id"].astype(str) == sid
    if mask.any():
        for col in FIELDS:
            updated.loc[mask, col] = student[col]
        action = "updated"
    else:
        updated = pd.concat([updated, pd.DataFrame([student])], ignore_index=True)
        action = "added"
    return updated, action

# Try by uncommenting and changing this sample:
# sample = {"student_id":"STU111","name":"Demo Learner","department":"CSE",
#           "graduation_year":2027,"cgpa":8.1,"status":"Applied",
#           "company":"—","package_lpa":np.nan}
# df, action = upsert_student(df, sample)
# print("Record", action)
# display(df.tail())

## 7. File handling — save and load CSV
This saves the tracker as a file in the Colab session. Download it using the next cell.

In [ ]:
DATA_FILE = "campus_compass_placement_tracker.csv"

def save_tracker(data, filepath=DATA_FILE):
    data.to_csv(filepath, index=False)  # file writing
    return os.path.abspath(filepath)

def load_tracker(filepath=DATA_FILE):
    loaded = pd.read_csv(filepath)      # file reading
    loaded["cgpa"] = pd.to_numeric(loaded["cgpa"], errors="coerce")
    loaded["package_lpa"] = pd.to_numeric(loaded["package_lpa"], errors="coerce")
    return loaded

saved_path = save_tracker(df)
print("Saved:", saved_path)
loaded_df = load_tracker()
print("Loaded rows:", len(loaded_df))
display(loaded_df.head())

In [ ]:
# Download CSV to your device (Google Colab)
from google.colab import files
files.download(DATA_FILE)

## 8. Export a company-wise summary


In [ ]:
company_summary = (
    df[df["status"] == "Placed"]
    .groupby("company")
    .agg(Placed_Students=("student_id", "count"),
         Average_Package_LPA=("package_lpa", "mean"),
         Highest_Package_LPA=("package_lpa", "max"))
    .round(2)
    .sort_values("Placed_Students", ascending=False)
)
display(company_summary)
company_summary.to_csv("company_placement_summary.csv")
print("Summary exported to company_placement_summary.csv")

## 9. Optional: download the company summary


In [ ]:
files.download("company_placement_summary.csv")

## 10. Viva / project explanation
**How the required concepts are used**
1. **List:** `records` stores multiple student records.
2. **Tuple:** `FIELDS` stores the fixed schema.
3. **Set:** unique departments and companies are identified without duplicates.
4. **Dictionary:** each student record uses key-value pairs; `student_lookup` supports ID lookup.
5. **Pandas:** DataFrame, filtering, grouping, CSV import/export.
6. **NumPy:** numeric arrays and missing-value-aware CGPA average.
7. **Files:** CSV records and company report are saved and loaded.

**Possible improvements**
- Add a form-based UI with dropdowns and validation (ipywidgets).
- Add login/role permissions before handling real student data.
- Use a database for multi-user deployment; CSV is for this learning prototype.
- Include offer count separately from placed-student count if students may receive multiple offers.

**Important metric note:** If your college counts placement rate using eligible students rather than all students, update the formula and label it clearly.